# FID analysis

Loads `results/fid_scores.json` and exports ranked tables.

In [1]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from gen_cats.evaluation.report_analysis import (
    MODEL_LABELS,
    SNIPPETS_DIR,
    best_runs_table,
    ensure_plots_dir,
    latex_fid_table,
    load_fid_scores,
    write_snippet,
    write_stats_csv,
)

sns.set_theme(style="whitegrid", context="talk")
PLOTS = ensure_plots_dir("results")
fid_data = load_fid_scores()
summary = best_runs_table(fid_data)
df = pd.DataFrame(summary)
df

,model,label,mean_fid,slug,hyperparameters,n_runs
0,beta_vae,$\beta$-VAE,278.290547,996f47caca39,"{'latent_dim': 128, 'beta': 1.0}",12
1,vqvae,VQ-VAE-1,243.510468,3acd0306b276,"{'num_embeddings': 512, 'feature_map_size': 8,...",24
2,pixelcnn,PixelCNN,267.631482,099fc9ce2263,"{'latent_dim': 128, 'beta': 1.0, 'num_embeddin...",3
3,tiny_ldm,Tiny LDM,NaN,,{},0
4,wgan_gp,WGAN-GP,174.402621,2b449afde8bc,"{'n_critic': 5, 'batch_size': 128, 'lr': 0.000...",24
5,sn_gan,SN-GAN,226.131552,737a5766e176,"{'n_critic': 1, 'batch_size': 128, 'lr': 0.000...",24
6,ddim,DDIM,NaN,,{},0


In [2]:
ranked = df.dropna(subset=["mean_fid"]).sort_values("mean_fid")
ranked[["label", "mean_fid", "slug", "n_runs"]]

,label,mean_fid,slug,n_runs
4,WGAN-GP,174.402621,2b449afde8bc,24
5,SN-GAN,226.131552,737a5766e176,24
1,VQ-VAE-1,243.510468,3acd0306b276,24
2,PixelCNN,267.631482,099fc9ce2263,3
0,$\beta$-VAE,278.290547,996f47caca39,12


In [3]:
plot_df = ranked.copy()
fig, ax = plt.subplots(figsize=(9, 4.5))
bars = ax.barh(plot_df["label"], plot_df["mean_fid"])
ax.invert_yaxis()
ax.set_xlabel("FID (lower is better)")
ax.set_title("Best grid cell per model family")
for bar, val in zip(bars, plot_df["mean_fid"], strict=True):
    ax.text(val + 3, bar.get_y() + bar.get_height() / 2, f"{val:.1f}", va="center", fontsize=11)
fig.tight_layout()
out = PLOTS / "fid_best_runs.png"
fig.savefig(out, dpi=200, bbox_inches="tight")
plt.close(fig)
out

PosixPath('/Users/pawelp/Desktop/education/pw/deepl/gen-cats/notebooks/plots/results/fid_best_runs.png')

In [6]:
rows = []
for entry in fid_data:
    model = entry["model"]
    for run in entry.get("runs", []):
        for seed, fid in run.get("per_seed", {}).items():
            rows.append(
                {
                    "model": MODEL_LABELS.get(model, model),
                    "fid": fid,
                    "seed": int(seed),
                    "slug": run.get("slug", ""),
                }
            )
all_df = pd.DataFrame(rows)
fig, ax = plt.subplots(figsize=(10, 4.5))
sns.boxplot(data=all_df, x="model", y="fid", ax=ax, width=0.6, legend=False)
sns.stripplot(data=all_df, x="model", y="fid", ax=ax, color="0.25", size=4, alpha=0.6)
ax.set_ylabel("FID")
ax.set_xlabel("")
plt.xticks(rotation=25, ha="right")
fig.tight_layout()
out = PLOTS / "fid_grid_distribution.png"
fig.savefig(out, dpi=200, bbox_inches="tight")
plt.close(fig)
all_df.describe()

,fid,seed
count,87.000000,87.000000
mean,282.925088,1149.666667
std,37.033205,1605.521604
min,174.402621,0.000000
25%,259.264439,0.000000
50%,282.499894,42.000000
75%,306.179593,3407.000000
max,372.247619,3407.000000


In [5]:
write_snippet("fid_table.tex", latex_fid_table(summary))
write_stats_csv(ranked[["model", "label", "mean_fid", "slug", "n_runs"]], "fid_best_runs.csv")
write_stats_csv(all_df, "fid_all_evals.csv")
list(SNIPPETS_DIR.glob("fid_*"))

[PosixPath('/Users/pawelp/Desktop/education/pw/deepl/gen-cats/notebooks/report_snippets/fid_best_runs.csv'),
 PosixPath('/Users/pawelp/Desktop/education/pw/deepl/gen-cats/notebooks/report_snippets/fid_mlflow_join.csv'),
 PosixPath('/Users/pawelp/Desktop/education/pw/deepl/gen-cats/notebooks/report_snippets/fid_all_evals.csv'),
 PosixPath('/Users/pawelp/Desktop/education/pw/deepl/gen-cats/notebooks/report_snippets/fid_table.tex')]